# Notebook 17 — Project Finalization
**MVP First | GitHub README · Architecture Diagram · Experiment Summary · Interview Preparation**

This notebook **does not retrain models**. It reads saved JSON reports from Notebooks 04–16 and exports a GitHub-ready documentation pack. It is intentionally runnable without raw RetailRocket data or model checkpoints.

Run with your Windows Conda `torch` kernel. Required: `pandas`, `matplotlib`, `nbformat` (optional for saving this notebook, not runtime).

In [ ]:
from pathlib import Path
import json, csv, shutil, zipfile
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not list(ROOT.glob("notebook16_cascade_report.json")):
    for candidate in [ROOT.parent, ROOT / "data" / "reports", ROOT.parent / "data" / "reports"]:
        if (candidate / "notebook16_cascade_report.json").exists():
            ROOT = candidate
            break
print("Report folder:", ROOT.resolve())
OUT = ROOT / "notebook17_project_finalization"
OUT.mkdir(parents=True, exist_ok=True)
REPORTS = {}
for n in range(4,17):
    matches = sorted(ROOT.glob(f"notebook{n:02d}*report.json"))
    if not matches:
        raise FileNotFoundError(f"Missing notebook{n:02d} report JSON in {ROOT}")
    REPORTS[n] = json.loads(matches[0].read_text(encoding="utf-8"))
print("Loaded", len(REPORTS), "report files")

In [ ]:
def row(n, field, method, k=20, split="test"):
    return next(x for x in REPORTS[n][field]
                if x.get("split")==split and x.get("method")==method and x.get("k")==k)

specs = [
    ("Popularity (decayed)",4,"metrics","decayed"),
    ("ItemCF (cooccurrence)",5,"metrics","cooccurrence"),
    ("SVD",6,"metrics","svd"),
    ("FAISS IVF nprobe16",7,"recommendation_metrics","faiss_ivf_np16"),
    ("Weighted RRF",8,"metrics","fusion_rrf"),
    ("LightGBM Binary",10,"ranking_metrics","binary"),
    ("LightGBM LambdaRank",10,"ranking_metrics","lambdarank"),
    ("DeepFM",11,"ranking_metrics","deepfm"),
    ("DeepFM + MMR",13,"test_metrics","mmr_0.90"),
    ("Cascade Top100 + MMR",16,"test_metrics","cascade_selected"),
]
quality=[]
for name,n,field,method in specs:
    x=row(n,field,method)
    quality.append(dict(stage=name,notebook=n,split="test",k=20,
        recall_at20=x["recall_micro"],ndcg_at20=x.get("ndcg"),
        hit_rate_at20=x.get("hit_rate"),ild_at20=x.get("intra_list_diversity"),
        category_coverage_at20=x.get("category_coverage"),
        note="Different retrieval/ranking stages; compare with care"))
quality_df=pd.DataFrame(quality)
quality_df.to_csv(OUT/"experiment_quality_summary.csv",index=False)
display(quality_df[["stage","recall_at20","ndcg_at20","ild_at20","category_coverage_at20"]])

In [ ]:
v=pd.DataFrame([dict(split="validation",keep=x["keep"],method=x["method"],
    recall_at20=x["recall_micro"],ndcg_at20=x["ndcg"],
    ild_at20=x["intra_list_diversity"],category_coverage_at20=x["category_coverage"])
    for x in REPORTS[16]["validation_metrics"] if x["k"]==20])
t=pd.DataFrame([dict(split="test",method=x["method"],keep=x["keep"],
    recall_at20=x["recall_micro"],ndcg_at20=x["ndcg"],
    ild_at20=x["intra_list_diversity"],category_coverage_at20=x["category_coverage"],
    missing_category_rate=x["missing_category_rate"])
    for x in REPORTS[16]["test_metrics"] if x["k"]==20])
v.to_csv(OUT/"validation_cascade_tradeoff.csv",index=False)
t.to_csv(OUT/"test_cascade_comparison.csv",index=False)
display(v)
display(t)
selected=int(REPORTS[16]["validation_selection"]["selected_keep"])
assert selected==100, f"README template is written for keep=100; got {selected}"
assert REPORTS[16]["validation_selection"]["test_used_for_selection"] is False
assert REPORTS[16]["serving_audit"]["top20_score_replay_exact_rate"]==1.0

In [ ]:
d14=next(x for x in REPORTS[14]["request_latency_qps"] if x["mode"]=="model_inference_replay")
d15=REPORTS[15]["http_latency"]
d16=REPORTS[16]["benchmark"]["http"]
latency=pd.DataFrame([
    dict(notebook=14,mode="model_inference_replay (in-process)",p50_ms=d14["p50_ms"],p95_ms=d14["p95_ms"],p99_ms=d14["p99_ms"],sequential_qps=d14["sequential_qps"],timed_requests=d14["n_requests"]),
    dict(notebook=15,mode="localhost HTTP sequential",p50_ms=d15["p50_ms"],p95_ms=d15["p95_ms"],p99_ms=d15["p99_ms"],sequential_qps=d15["sequential_qps"],timed_requests=d15["n"]),
    dict(notebook=16,mode="localhost HTTP sequential",p50_ms=d16["p50_ms"],p95_ms=d16["p95_ms"],p99_ms=d16["p99_ms"],sequential_qps=d16["sequential_qps"],timed_requests=d16["n"])
])
latency.to_csv(OUT/"serving_latency_summary.csv",index=False)
display(latency)
print("Note: N14 in-process is NOT directly comparable to N15/N16 HTTP.")

In [ ]:
fig, ax=plt.subplots(figsize=(10,5))
view=quality_df.dropna(subset=["ndcg_at20"])
ax.barh(view["stage"],view["recall_at20"]*100)
ax.invert_yaxis()
ax.set_xlabel("Test Recall@20 (%)")
ax.set_title("Ranking pipeline: offline test recall (different stages)")
fig.tight_layout()
fig.savefig(OUT/"quality_comparison.png",dpi=160)
plt.show()

fig,ax=plt.subplots(figsize=(8,4))
for col in ["recall_at20","ndcg_at20"]:
    ax.plot(v["keep"],v[col],marker="o",label=col)
ax.set_xlabel("LightGBM candidates retained")
ax.set_title("Validation-only cascade sweep (metrics have different scales)")
ax.legend()
fig.tight_layout()
fig.savefig(OUT/"validation_tradeoff.png",dpi=160)
plt.show()

fig,ax=plt.subplots(figsize=(7,4))
http=latency[latency.notebook.isin([15,16])]
ax.bar([str(x) for x in http.notebook],http.p50_ms)
ax.set_xlabel("Notebook (HTTP benchmark)")
ax.set_ylabel("P50 latency (ms)")
ax.set_title("Local sequential HTTP P50")
fig.tight_layout()
fig.savefig(OUT/"http_latency_comparison.png",dpi=160)
plt.show()

In [ ]:
ARCHITECTURE = 'flowchart TD\n    A["RetailRocket implicit events<br/>views / add-to-cart / transactions"] --> B["Time-based train / validation / test split"]\n    B --> C["Offline user/item/category features"]\n    B --> D1["Popularity"]\n    B --> D2["ItemCF"]\n    B --> D3["SVD embeddings + FAISS"]\n    C --> D4["Category-aware retrieval"]\n    D1 --> E["Weighted RRF Fusion<br/>Top 500 candidates"]\n    D2 --> E\n    D3 --> E\n    D4 --> E\n    E --> F["LightGBM pre-ranker<br/>Top 500 → Top 100"]\n    C --> F\n    F --> G["DeepFM fine-ranker<br/>100 candidates"]\n    C --> G\n    G --> H["MMR re-ranking<br/>lambda = 0.9"]\n    H --> I["FastAPI /recommend<br/>Top 20"]\n    B --> J["Offline Recall / NDCG<br/>ILD / Coverage"]\n    I --> K["Parity audit + local HTTP latency"]\n    I -. "Not implemented" .-> L["Live event retrieval / feature store"]\n'
README_TEMPLATE = '# Multi-Stage Personalized Recommendation System\n\n**Portfolio project | RetailRocket implicit-feedback dataset | Python · DuckDB · FAISS · LightGBM · PyTorch · FastAPI**\n\n> **Scope:** reproducible offline recommendation pipeline with a local HTTP demo. **Not** a production ads platform or an online CTR/CVR estimator. Retrieval candidates and user/item features are precomputed; LightGBM and DeepFM inference run inside the service.\n\n## Highlights\n\n- Time-based evaluation and reproducible feature engineering on implicit event logs.\n- Multi-source retrieval: popularity, item collaborative filtering, SVD/FAISS and category-aware candidates, fused with weighted reciprocal rank fusion (RRF).\n- Two-stage ranking: **LightGBM Top-500 → Top-100**, then **DeepFM Top-100**, followed by **MMR (λ = 0.9)**.\n- Local FastAPI `/recommend` endpoint, model warm-loading, validation-only cascade selection and offline-to-serving parity audit.\n- Trade-offs measured across relevance, diversity, coverage, and local sequential HTTP latency.\n\n## Architecture\n\n```mermaid\nflowchart TD\n    A["RetailRocket implicit events<br/>views / add-to-cart / transactions"] --> B["Time-based train / validation / test split"]\n    B --> C["Offline user/item/category features"]\n    B --> D1["Popularity"]\n    B --> D2["ItemCF"]\n    B --> D3["SVD embeddings + FAISS"]\n    C --> D4["Category-aware retrieval"]\n    D1 --> E["Weighted RRF Fusion<br/>Top 500 candidates"]\n    D2 --> E\n    D3 --> E\n    D4 --> E\n    E --> F["LightGBM pre-ranker<br/>Top 500 → Top 100"]\n    C --> F\n    F --> G["DeepFM fine-ranker<br/>100 candidates"]\n    C --> G\n    G --> H["MMR re-ranking<br/>lambda = 0.9"]\n    H --> I["FastAPI /recommend<br/>Top 20"]\n    B --> J["Offline Recall / NDCG<br/>ILD / Coverage"]\n    I --> K["Parity audit + local HTTP latency"]\n    I -. "Not implemented" .-> L["Live event retrieval / feature store"]\n```\n\n## Key results (RetailRocket test split)\n\n| Metric | Full-500 + MMR | Selected cascade Top-100 + MMR |\n|---|---:|---:|\n| Recall@20 | 7.414% | **7.572%** |\n| NDCG@20 | 0.055652 | **0.056089** |\n| Intra-list diversity@20 | **0.31154** | 0.29637 |\n| Average category coverage@20 | **4.781** | 4.320 |\n\n**Serving benchmark:** local Windows / Conda `torch`, single Uvicorn worker, 55 sequential timed requests after five warm-ups. Notebook 16 HTTP **P50 21.23 ms / P95 24.49 ms / P99 26.20 ms**, **46.40 sequential QPS**. No concurrency, network deployment, or cold-start claim. Model inference is real; retrieval/features are cached.\n\n**Validation-only decision:** tried pre-ranking keep={100, 150, 200, 300} against Full-500, with Recall@20 absolute drop ≤ 0.005 and NDCG@20 relative drop ≤ 5%; chose the smallest eligible keep, **100**. Test data was not used to select the keep count. Earlier upstream model and design iterations should not be represented as a fully untouched independent test protocol.\n\n## Reproduce locally (Windows / Conda)\n\n1. Download RetailRocket `events.csv`, `item_properties_part1.csv`, `item_properties_part2.csv`, `category_tree.csv` separately; **do not commit raw datasets or trained models**.\n2. Place data in the paths expected by Notebook 01 and use the `torch` Conda kernel.\n3. Run notebooks **01–16 in order**, following their own requirements and output paths. The notebooks produce processed Parquet files, model checkpoints, and `data/reports/` reports.\n4. For the final local service, place `notebook15_service.py`, `notebook16_service.py`, and `notebook16_requirements.txt` at the repository root. The Notebook 16 service uses components from Notebook 15.\n5. Install dependencies with `python -m pip install -r notebook16_requirements.txt` and follow Notebook 16\'s service startup and parity cells; its report contains the selected candidate count. Run Notebook 17 to regenerate this documentation and the summary tables.\n\n**API:** `GET /health` and `GET /recommend?user_id=<known_user_id>&k=20`. Use a user ID available in the cached test split; this is a closed-world offline demo.\n\n## Experiments\n\n- [`experiment_quality_summary.csv`](experiment_quality_summary.csv) — test Recall@20 and NDCG@20 across retrieval and ranking methods.\n- [`validation_cascade_tradeoff.csv`](validation_cascade_tradeoff.csv) — validation-only keep-count sweep.\n- [`test_cascade_comparison.csv`](test_cascade_comparison.csv) — final frozen test comparison.\n- [`serving_latency_summary.csv`](serving_latency_summary.csv) — Notebook 14/15/16 benchmark summary; note that Notebook 14 is **in-process**, while Notebook 15/16 use **HTTP**, so this is not a controlled apples-to-apples speedup study.\n- [`interview_preparation.md`](interview_preparation.md) — system story, trade-offs and likely interview questions.\n\n## Important limitations\n\n1. **Implicit feedback, no impression logs:** unobserved items are not reliable negatives; offline relevance is not CTR/CVR.\n2. **Retrieval ceiling:** recall is limited by the Top-500 candidate pool, and pruning trades diversity for computation.\n3. **Category metadata:** missing categories can bias ILD and coverage estimates.\n4. **No live retrieval or feature store:** cached test users and feature rows are used; cold-start and real-time updates are not implemented.\n5. **Small local latency sample:** 55 sequential requests are insufficient for production P99 or concurrent throughput estimates.\n6. **No online A/B tests:** all reported improvements are offline and descriptive, not causal business impact.\n\n## Repository layout (suggested)\n\n```text\nmulti-stage-ads-recommendation/\n├── notebooks/                    # Notebooks 01–17\n├── notebook15_service.py\n├── notebook16_service.py\n├── notebook16_requirements.txt\n├── data/                         # ignored: raw / processed / reports\n├── models/                       # ignored: checkpoints\n└── docs/\n    ├── architecture.mmd\n    ├── experiment_quality_summary.csv\n    ├── validation_cascade_tradeoff.csv\n    ├── test_cascade_comparison.csv\n    ├── serving_latency_summary.csv\n    └── interview_preparation.md\n```\n\n## Interview-ready summary\n\n> I built a multi-stage recommendation system on RetailRocket event data. I combined multiple retrieval sources using RRF, trained a LightGBM pre-ranker and a DeepFM fine-ranker, and applied MMR to improve category diversity. I used a time-based split, selected the cascade size on validation only, and served the pipeline through FastAPI. Keeping only 100 of 500 candidates preserved test Recall@20 while reducing the DeepFM workload by 80%. The local sequential HTTP benchmark reached around 21 ms P50. The main limitation is that retrieval and features are precomputed, so I do not claim production-scale serving or online CTR lift.\n'
INTERVIEW_TEMPLATE = '# Interview Preparation — Multi-Stage Recommendation System\n\n## 30-second pitch (straightforward spoken English)\n\nI built an end-to-end recommendation portfolio project using RetailRocket behavior data. The system combines several retrieval methods, then uses LightGBM for pre-ranking, DeepFM for fine-ranking, and MMR for diversity. I also deployed a local FastAPI endpoint and checked that its output matched the offline pipeline. My main focus was the trade-off between recommendation quality, latency, and engineering complexity.\n\n## 2-minute project walkthrough\n\n**Problem:** Rank relevant products for users from sparse implicit actions. The dataset has user events, not impression or click-through logs, so this is an offline next-interaction ranking task rather than true ad CTR prediction.\n\n**Data:** I used a time-based train/validation/test split and built user, item, and category features using DuckDB. I was careful about data leakage and separated historical features from future labels.\n\n**Retrieval:** I combined popularity, item-based collaborative filtering, SVD/FAISS, and category-aware candidates with weighted reciprocal rank fusion. The fused stage returns 500 candidates per user.\n\n**Ranking:** LightGBM scores all 500 and keeps 100; DeepFM scores the 100 survivors; MMR reorders the results using a frozen λ of 0.9. The keep count was selected using validation data only.\n\n**Evaluation:** On the test split, Recall@20 moved from 7.414% for the full-500 MMR reference to 7.572% for the selected cascade, while NDCG@20 moved from 0.05565 to 0.05609. Category diversity and coverage decreased, so the change is a trade-off, not a universal improvement.\n\n**Serving:** I exposed the cascade with FastAPI. Models run inference in the server, but retrieval and feature values are precomputed. On 55 local sequential timed HTTP requests, P50 was 21.23 ms. Serving-to-offline cascade parity was 100% on 60 sampled users.\n\n**What I would do next:** Add live feature freshness, a more realistic impression-based dataset, concurrent load tests, diversity constraints, and online A/B testing if production traffic were available.\n\n## Questions and concise answers\n\n**Q1. Why use multi-stage ranking instead of one model?**  \nA: Expensive models should not score the whole catalog. Retrieval reduces millions of possible items to hundreds, LightGBM filters them, and DeepFM spends computation on the most promising candidates.\n\n**Q2. Why weighted RRF?**  \nA: Retrieval sources produce scores on different scales. RRF combines their ranks without needing perfectly calibrated scores.\n\n**Q3. Why LightGBM before DeepFM?**  \nA: LightGBM is a strong tabular baseline and a useful pruning stage. DeepFM models feature interactions, but running it on fewer candidates reduces work.\n\n**Q4. Why did pruning slightly improve Recall@20?**  \nA: Removing low-quality candidates can change the final competition for the Top-20. The measured gain is small and could reflect sample variation; it does not prove pruning is intrinsically better.\n\n**Q5. What is the biggest quality trade-off?**  \nA: Test category coverage@20 dropped from 4.781 to 4.320 and ILD@20 from 0.3115 to 0.2964. I would add a diversity guardrail before using this in a product.\n\n**Q6. Is 7.57% Recall@20 good?**  \nA: It depends on the candidate pool, label definition, sparsity, and dataset. I would compare against baselines on the same split, not against unrelated commercial CTR numbers.\n\n**Q7. Why not report CTR or conversion rate?**  \nA: The data does not contain impression exposure logs. Unobserved interactions are not known negatives, so I report offline ranking metrics rather than calibrated online CTR.\n\n**Q8. How did you avoid leakage?**  \nA: I used time cutoffs and computed historical features from data available before the label period. I selected the cascade size using validation only. I would still audit all earlier modeling decisions before claiming the final test was fully untouched.\n\n**Q9. How do you measure inference consistency?**  \nA: I replay the same cached user candidates and features through the service and compare ordered Top-20 IDs with the offline cascade output.\n\n**Q10. Is your HTTP latency production-ready?**  \nA: No. It is a localhost, single-worker, sequential warm benchmark with 55 measured requests. It excludes live retrieval, feature generation, network conditions, cold starts, and concurrency.\n\n**Q11. Why use MMR?**  \nA: Pure relevance ranking can show near-duplicate categories. MMR balances relevance with item dissimilarity, but it may reduce Recall or NDCG.\n\n**Q12. What would you optimize first?**  \nA: I would profile the real end-to-end service under concurrent traffic, then optimize feature conversion and MMR. I would also improve candidate retrieval, because downstream ranking cannot recover missed items.\n\n**Q13. What was your hardest engineering problem?**  \nA: Maintaining feature schema and score parity between offline training and online model inference, especially category vocabularies and preprocessing.\n\n**Q14. Why does DeepFM not always outperform LightGBM?**  \nA: Tree models can be very strong on engineered tabular features. Deep models need enough signal, carefully tuned sampling, and regularization. More complex is not automatically better.\n\n**Q15. What is the difference between recall and NDCG?**  \nA: Recall measures how many relevant future interactions appear in the Top-K. NDCG also rewards placing relevant items higher.\n\n## Honest resume bullets (adapt to your actual ownership)\n\n- Built a reproducible multi-stage recommendation pipeline on RetailRocket implicit events using DuckDB, FAISS, LightGBM, DeepFM, and MMR, with temporal evaluation and offline ranking metrics.\n- Implemented a validation-selected Top-500 → Top-100 cascade, reducing DeepFM candidate volume by 80% while achieving 7.57% test Recall@20 and 0.0561 NDCG@20.\n- Served precomputed-candidate recommendations through FastAPI with real LightGBM/DeepFM inference, 100% sampled offline-to-serving Top-20 parity, and 21.23 ms local sequential HTTP P50 latency.\n\n## What not to claim\n\n- Do not claim online CTR lift, A/B-test improvement, production QPS, live retrieval, real-time features, causal business impact, or broad generalization from 55 timed requests.\n'
(OUT/'architecture.mmd').write_text(ARCHITECTURE,encoding='utf-8')
(OUT/'README.md').write_text(README_TEMPLATE,encoding='utf-8')
(OUT/'interview_preparation.md').write_text(INTERVIEW_TEMPLATE,encoding='utf-8')
print('Markdown and Mermaid files written')

## Release checklist

- [ ] Confirm every number against its JSON report.
- [ ] Add real screenshots of the API response and notebook plots to the GitHub repository.
- [ ] Put raw datasets, `.duckdb`, `.parquet`, checkpoints, and `.env` in `.gitignore`.
- [ ] Use your own GitHub repository URL (none is fabricated here).
- [ ] Verify README Quick Start in a fresh Conda environment.
- [ ] Practice the 30-second and 2-minute English project pitches.
- [ ] Explicitly mention the **precomputed retrieval/features** limitation.

In [ ]:
required=["README.md","architecture.mmd","interview_preparation.md",
          "experiment_quality_summary.csv","validation_cascade_tradeoff.csv",
          "test_cascade_comparison.csv","serving_latency_summary.csv",
          "quality_comparison.png","validation_tradeoff.png","http_latency_comparison.png"]
assert all((OUT/x).exists() for x in required)
manifest={"notebook":17,"source_report_notebooks":list(REPORTS),
          "selected_keep":selected,"source":"saved offline JSON reports",
          "outputs":required,"note":"Documentation only; no model training or new evaluation"}
(OUT/"notebook17_finalization_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
with zipfile.ZipFile(ROOT/"notebook17_project_finalization.zip","w",zipfile.ZIP_DEFLATED) as z:
    for f in OUT.iterdir():
        if f.is_file(): z.write(f,arcname=f.name)
print("Release pack:",ROOT/"notebook17_project_finalization.zip")
print("Files:",sorted(p.name for p in OUT.iterdir()))